In [1]:
import os

import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2

In [2]:
# Basic experiment settings
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)


cudnn.deterministic = True
cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Declare directories
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

# Experiment parameters
img_size = 384
batch_size = 16
num_workers = 4
num_classes = 5

cpu


In [3]:
# Currently testing vision transformers so check its architecture
vit_model = torch.load("/kaggle/input/vit-new-base/vit.pt")
vit_model.to(device)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/vit-new-base/vit.pt'

In [4]:
# Test dataset without labels
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava data.

    Args:
        data_dir: base directory to the images.
        img_label_df: dataframe containing image-label pairs.
        transforms: set of transforms to be used.
    """

    def __init__(self, data_dir, transform=None, ttas=None, img_size=384):
        super().__init__(root=data_dir)

        self.transform = transform
        self.images = os.listdir(data_dir)
        self.ttas = ttas

    def __getitem__(self, idx):
        filename = self.images[idx]

        img = Image.open(os.path.join(self.root, filename))
        img = [self.transform(t(img)) for t in self.ttas]

        return img, filename


    def __len__(self):
        return len(self.images)

In [5]:
# Create the set of transforms for test time
test_transforms = v2.Compose(
    [
        v2.Resize((img_size, img_size), interpolation=InterpolationMode.BICUBIC),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

ttas = [
    v2.RandomResizedCrop((img_size, img_size), (0.5, 1)),
    v2.RandomRotation(180),
    v2.RandomVerticalFlip(1),
    v2.RandomAffine(180),
    v2.RandomPerspective(p=1),
]

# Dataset and dataloader
test_dataset = CassavaDataset(test_dir, transform=test_transforms, ttas=ttas)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

# Adjust unnormalized outputs
normalizer = torch.nn.Softmax(dim=1)

In [6]:
# Perform inference
all_names = []
all_preds = []

vit_model.eval()

with torch.no_grad():
    test_loss = 0

    for batch_idx, (inputs, filenames) in enumerate(test_loader):
        inputs, filenames = torch.cat(inputs, dim=0).to(device), list(filenames)
        batch_size = len(filenames)
        
        # Now do the input predictions
        preds = normalizer(vit_model(inputs))

        # Find the average prediction by each model
        batch_preds = torch.stack(torch.split(preds, len(filenames)), dim=0)
        mean_preds = torch.mean(batch_preds, dim=0)
        pred_labels = torch.argmax(mean_preds, 1).tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)

NameError: name 'vit_model' is not defined

In [7]:
# Create submission
my_submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
my_submission.to_csv("submission.csv", index=False)